# Stemming

En este notebook usamos stemming como técnica de normalización del texto antes de vectorizarlo. A diferencia de la lematización, el stemming no busca la forma canónica del diccionario sino que recorta los sufijos de cada palabra según reglas morfológicas. El resultado es más agresivo pero también más rápido y sin necesidad de un modelo lingüístico externo. Usamos el `SnowballStemmer` de NLTK para español.

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

from nltk.stem import SnowballStemmer

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## 2. Carga de los datos

In [ ]:
df = pd.read_csv('train.csv')
df_eval = pd.read_csv('eval.csv')

In [ ]:
df.head()

In [ ]:
df.shape

## 3. Exploración del conjunto de datos

In [ ]:
df['decade'].value_counts().sort_index().plot(
    kind='bar', figsize=(14, 4), title='Distribución de décadas'
)
plt.xlabel('Década')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

In [ ]:
def reporte_calidad(df):
    reporte_de_cualidad = {
        'Total records': len(df),
        'duplicated record': df.duplicated().sum(),
        'missing values': df.isnull().sum().to_dict(),
        'data types': df.dtypes.astype(str).to_dict(),
    }
    return reporte_de_cualidad

print(reporte_calidad(df))

## 4. Preprocesamiento del texto

Aplicamos stemming con el algoritmo Snowball para español. Primero limpiamos el texto (minúsculas, eliminación de caracteres no alfabéticos) y luego aplicamos el stemmer a cada token. El resultado se une de nuevo en una cadena para que el vectorizador lo procese normalmente.

In [ ]:
stemmer = SnowballStemmer('spanish')

def limpiar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r'[^a-záéíóúüñ\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def stemear(texto):
    tokens = limpiar_texto(texto).split()
    return ' '.join([stemmer.stem(t) for t in tokens])

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df['texto_stem'] = df['text'].apply(stemear)
df[['text', 'texto_stem', 'decade']].head()

## 5. Partición de los datos

In [ ]:
X = df['texto_stem']
y = df['decade']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

Usamos `stratify=y` para mantener la distribución de clases.

In [ ]:
X_train.shape, X_val.shape

## 6. Construcción del pipeline

El stemming se aplicó antes de la partición. El pipeline encadena directamente TF-IDF con el clasificador.

In [ ]:
pipeline_stem = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=1000, solver='saga')),
])

## 7. Entrenamiento con búsqueda de hiperparámetros

Usamos el mismo espacio de búsqueda que en el notebook de lematización para facilitar la comparación directa entre ambos enfoques de normalización.

In [ ]:
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__max_features': [20000, 50000],
    'tfidf__min_df': [1, 2],
    'clf__C': [0.1, 1, 10],
}

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=0)
grid_stem = GridSearchCV(
    pipeline_stem, param_grid, cv=kfold, scoring='accuracy', n_jobs=-1
)

In [ ]:
grid_stem.fit(X_train, y_train)

In [ ]:
print('Mejores hiperparámetros:', grid_stem.best_params_)
print('Mejor score CV (accuracy):', round(grid_stem.best_score_, 4))

## 8. Evaluación del mejor modelo

In [ ]:
best_model = grid_stem.best_estimator_

y_pred_train = best_model.predict(X_train)
y_pred_val   = best_model.predict(X_val)

#### Comparación de rendimientos sobre entrenamiento y validación

In [ ]:
print('Accuracy en entrenamiento:', round(accuracy_score(y_train, y_pred_train), 4))
print('Accuracy en validación:   ', round(accuracy_score(y_val,   y_pred_val),   4))
print('Mejor score CV (accuracy):', round(grid_stem.best_score_,                 4))

In [ ]:
print(classification_report(y_val, y_pred_val))

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))
ConfusionMatrixDisplay.from_predictions(y_val, y_pred_val, ax=ax, colorbar=False)
plt.title('Matriz de confusión — validación')
plt.tight_layout()
plt.show()

## 9. Predicciones sobre eval.csv

In [ ]:
df_eval['texto_stem'] = df_eval['text'].apply(stemear)

y_eval_pred = best_model.predict(df_eval['texto_stem'])

submission = pd.DataFrame({'id': df_eval['id'], 'answer': y_eval_pred})
submission.to_csv('submission_stemming.csv', index=False)
submission.head()